# Week 08 — Home exercise 5: A merge that goes wrong

**Solution proposal.**

Diagnosing a join that runs perfectly and quietly deletes 48 rows.

In [1]:
import pandas as pd

co2 = pd.read_csv("../data/co2_emissions.csv")
info = pd.read_csv("../data/country_info.csv")

print("emissions:", co2.shape)
print("lookup:   ", info.shape)

emissions: (6240, 11)
lookup:    (295, 4)


## 1. The inner join

In [2]:
inner = co2.merge(info, left_on="country", right_on="name", how="inner")

print("rows before:", len(co2))
print("rows after: ", len(inner))
print("lost:       ", len(co2) - len(inner))

rows before: 6240
rows after:  6192
lost:        48


**48 rows gone**, and no error, no warning, nothing in the output of the merge itself that says so.
The only reason we know is that we counted.

48 out of 6 240 is 0.8%. It would survive a `.head()`, a `.describe()`, a plot, and a conversation
with your supervisor.

## 2. The same merge as a left join, with `indicator=True`

In [3]:
checked = co2.merge(
    info,
    left_on="country",
    right_on="name",
    how="left",
    indicator=True,
)

print("rows:", len(checked))
checked["_merge"].value_counts()

rows: 6240


_merge
both          6192
left_only       48
right_only       0
Name: count, dtype: int64

Nothing is deleted now, and the `_merge` column says what happened to each row: **5 856 matched, 48
did not**, and nothing came from the right side alone.

The 48 add up exactly with the inner join's loss, which confirms the diagnosis rather than assuming
it.

## 3. Which entities, and why

In [4]:
unmatched = checked[checked["_merge"] == "left_only"]["country"].unique()

print(list(unmatched))
print("years each:", len(checked[checked["country"] == unmatched[0]]))

['Latin America & Caribbean', 'Sub-Saharan Africa']
years each: 24


Two entities, 24 years apiece. And they are obviously in the lookup file — you can see them.

In [5]:
print("present in the lookup file?")
for name in unmatched:
    matches = (info["name"].str.strip() == name).sum()
    print(" ", name, "->", matches, "match after stripping")

present in the lookup file?
  Latin America & Caribbean -> 1 match after stripping
  Sub-Saharan Africa -> 1 match after stripping


So the entity **is** there, and the merge still failed. That rules out "missing data" and points at
the key itself. Print the lookup file's version of the string the way Python writes it, with `repr`,
which shows the quotes and therefore shows the edges.

In [6]:
suspects = info[info["name"].str.strip().isin(unmatched)]

for name in suspects["name"]:
    print(repr(name), "  length:", len(name), " stripped length:", len(name.strip()))

'Latin America & Caribbean '   length: 26  stripped length: 25
'Sub-Saharan Africa '   length: 19  stripped length: 18


**A single trailing space.** `'Sub-Saharan Africa '` is 19 characters and `'Sub-Saharan Africa'` is
18, and no display pandas produces will ever show you the difference.

This is exactly how the World Bank publishes the file, so it is not a corruption to be reported —
it is a property of real data to be handled.

## 4. Two fixes

**Fix A: clean the key.** `.str.strip()` on both sides, so that whatever whitespace either file
carries is gone before the comparison.

In [7]:
co2_clean = co2.copy()
info_clean = info.copy()

co2_clean["country"] = co2_clean["country"].str.strip()
info_clean["name"] = info_clean["name"].str.strip()

fix_a = co2_clean.merge(info_clean, left_on="country", right_on="name", how="left")

print("rows:", len(fix_a), "| region missing:", fix_a["region"].isna().sum())

rows: 6240 | region missing: 0


**Fix B: use a key that was never dirty.** Both files carry the ISO three-letter `code`, which has no
whitespace, no capitalization question and no spelling variants.

In [8]:
fix_b = co2.merge(info[["code", "region", "incomeLevel"]], on="code", how="left")

print("rows:", len(fix_b), "| region missing:", fix_b["region"].isna().sum())

rows: 6240 | region missing: 0


Both give 6 240 rows with nothing missing.

### Which would I use?

**Here, `code`** — and it is not close. A code is designed to be a key: it is short, it has one
canonical form, and it cannot be misspelled without becoming obviously invalid. A country name is
designed to be read by humans, and it varies by whitespace, by capitalization, by punctuation
(`Cote d'Ivoire`), by language, and by politics (`Korea, Rep.` against `South Korea`).

**On a file with no code column**, you have no choice, and then `.str.strip()` on both sides before
merging is the minimum. Realistically you would also `.str.lower()` for the comparison, and then check
what still failed to match with `indicator=True` — because whitespace is the *easiest* of the problems
a name key has, and fixing it just reveals the next one.

The general rule: **merge on the most machine-made column available.** Prefer a code to a name, an id
to a code, and a name only when nothing else exists.

## 5. `validate=`

In [9]:
panel = co2.merge(
    info[["code", "region", "incomeLevel"]],
    on="code",
    how="left",
    validate="many_to_one",
)

print("validated, rows:", len(panel))

validated, rows: 6240


It held. Now claim the wrong shape on purpose.

In [10]:
try:
    co2.merge(info[["code", "region"]], on="code", how="left", validate="one_to_one")
except Exception as error:
    print(type(error).__name__)
    print(str(error).split("\n")[0])

MergeError
Merge keys are not unique in left dataset; not a one-to-one merge


`one_to_one` is false because the emissions file has 24 rows per country — which is true and obvious,
and is precisely the kind of thing people get wrong about data they did not create.

The value of `validate=` is that it turns a **belief** into a **check**. Writing `many_to_one` is
saying out loud "I think this lookup has one row per country", and if a future version of the file
gains a duplicate, the merge stops instead of silently doubling your rows.

## 6. The question from last week

In [11]:
countries = panel[panel["region"] != "Aggregates"]

print("entities, all:      ", panel["code"].nunique())
print("entities, countries:", countries["code"].nunique())
print("rows:               ", len(panel), "->", len(countries))

entities, all:       260
entities, countries: 217
rows:                6240 -> 5208


In [12]:
year_2023 = countries[countries["year"] == 2023]

year_2023.sort_values("co2_total", ascending=False)[["country", "code", "co2_total"]].head(10).round(1)

,country,code,co2_total
1127,China,CHN,13021.2
5975,United States,USA,4618.3
2639,India,IND,3014.5
4751,Russian Federation,RUS,1948.3
2855,Japan,JPN,1002.4
2687,"Iran, Islamic Rep.",IRN,816.1
2663,Indonesia,IDN,772.6
4871,Saudi Arabia,SAU,635.4
2999,"Korea, Rep.",KOR,590.4
2111,Germany,DEU,589.7


**Now it is a list of countries.** China, the United States, India, Russia, Japan — a ranking you can
put in a report, instead of last week's list of five World Bank groupings.

43 of the 260 entities were aggregates. Every one of them was in last week's totals, averages and
rankings, and none of them announced itself.

### Things worth noticing

- **The diagnosis had four steps and only the last one was the fix.** Count, then `indicator=`, then
  find the rows, then look at the key with `repr`. Jumping straight to "use `code` instead" gets the
  right answer this time and teaches you nothing about the next file.
- **`repr()` is the tool for invisible characters.** Trailing spaces, tabs, non-breaking spaces and
  the odd byte-order mark all look like nothing at all until you print the quotes.
- **A left join plus `indicator=True` is non-destructive.** It is the right first move whenever a
  merge surprises you, because it lets you look at the failures instead of at their absence.

### What this notebook does NOT do

- It only finds the entities where the *emissions* file failed to match. `right_only` was zero here,
  but a lookup table with entries nobody uses is its own kind of problem, and the same
  `indicator=True` would have shown it.
- `.str.strip()` handles whitespace at the ends and nothing else. `"Cote d'Ivoire"` against
  `"Côte d'Ivoire"`, or `"Korea, Rep."` against `"South Korea"`, would still fail, and no amount of
  stripping helps — those need a lookup table of alternative names, which is a real and tedious part
  of this kind of work.
- Dropping every entity whose region is `"Aggregates"` trusts one column of one file completely. It is
  right here, and it is worth knowing that the whole result rests on it.